In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/telco_clean.csv")

print("Loaded:", df.shape)

Loaded: (7043, 21)


In [2]:
cols = ["tenure", "MonthlyCharges", "TotalCharges"]

print("ALL CUSTOMERS")
print(df[cols].agg(["mean", "median", "std"]).round(1))
print()
print("STAYED (0) vs LEFT (1)")
print(df.groupby("Churn")[cols].agg(["mean", "median", "std"]).round(1).T)

ALL CUSTOMERS
        tenure  MonthlyCharges  TotalCharges
mean      32.4            64.8        2279.7
median    29.0            70.4        1394.6
std       24.6            30.1        2266.8

STAYED (0) vs LEFT (1)
Churn                       0       1
tenure         mean      37.6    18.0
               median    38.0    10.0
               std       24.1    19.5
MonthlyCharges mean      61.3    74.4
               median    64.4    79.6
               std       31.1    24.7
TotalCharges   mean    2549.9  1531.8
               median  1679.5   703.6
               std     2330.0  1890.8


In [3]:
stayed = df[df["Churn"] == 0]["MonthlyCharges"]
left = df[df["Churn"] == 1]["MonthlyCharges"]

print("Average monthly charge - stayed:", round(stayed.mean(), 2))
print("Average monthly charge - left:  ", round(left.mean(), 2))

t_stat, p_value = stats.ttest_ind(left, stayed, equal_var=False)

print()
print("t-statistic:", round(t_stat, 2))
print("p-value:", p_value)
print()

if p_value < 0.05:
    print("Result: the difference is statistically significant.")
else:
    print("Result: no significant difference found.")

Average monthly charge - stayed: 61.27
Average monthly charge - left:   74.44

t-statistic: 18.41
p-value: 8.592449331547067e-73

Result: the difference is statistically significant.


In [4]:
cat_cols = ["Contract", "PaymentMethod", "InternetService", "TechSupport",
            "OnlineSecurity", "gender", "PhoneService"]

results = []
for col in cat_cols:
    table = pd.crosstab(df[col], df["Churn"])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    results.append({"feature": col, "chi2": round(chi2, 1), "p_value": p,
                    "related_to_churn": p < 0.05})

results_df = pd.DataFrame(results).sort_values("chi2", ascending=False)
print(results_df.to_string(index=False))

        feature   chi2       p_value  related_to_churn
       Contract 1184.6 5.863038e-258              True
 OnlineSecurity  850.0 2.661150e-185              True
    TechSupport  828.2 1.443084e-180              True
InternetService  732.3 9.571788e-160              True
  PaymentMethod  648.1 3.682355e-140              True
   PhoneService    0.9  3.387825e-01             False
         gender    0.5  4.865787e-01             False
